In [1]:
import pandas as pd
from pathlib import Path

print("Pandas version:", pd.__version__)

Pandas version: 2.2.2


In [2]:
DATA_DIR = Path("archive")

orders_file = DATA_DIR / "olist_orders_dataset.csv"
customers_file = DATA_DIR / "olist_customers_dataset.csv"
items_file = DATA_DIR / "olist_order_items_dataset.csv"
payments_file = DATA_DIR / "olist_order_payments_dataset.csv"

print("Checking files...")

for file in [orders_file, customers_file, items_file, payments_file]:
    print(file, "->", file.exists())

Checking files...
archive\olist_orders_dataset.csv -> True
archive\olist_customers_dataset.csv -> True
archive\olist_order_items_dataset.csv -> True
archive\olist_order_payments_dataset.csv -> True


In [3]:
DATA_DIR = Path("archive")

orders_file = DATA_DIR / "olist_orders_dataset.csv"
customers_file = DATA_DIR / "olist_customers_dataset.csv"
items_file = DATA_DIR / "olist_order_items_dataset.csv"
payments_file = DATA_DIR / "olist_order_payments_dataset.csv"

print("Checking files...")

for file in [orders_file, customers_file, items_file, payments_file]:
    print(file, "->", file.exists())

Checking files...
archive\olist_orders_dataset.csv -> True
archive\olist_customers_dataset.csv -> True
archive\olist_order_items_dataset.csv -> True
archive\olist_order_payments_dataset.csv -> True


In [5]:
orders = pd.read_csv(orders_file)
customers = pd.read_csv(customers_file)
items = pd.read_csv(items_file)
payments = pd.read_csv(payments_file)

print("Orders:", orders.shape)
print("Customers:", customers.shape)
print("Items:", items.shape)
print("Payments:", payments.shape)

Orders: (99441, 8)
Customers: (99441, 5)
Items: (112650, 7)
Payments: (103886, 5)


In [6]:
print("ORDERS")
print(orders.columns.tolist())

print("\nCUSTOMERS")
print(customers.columns.tolist())

print("\nITEMS")
print(items.columns.tolist())

print("\nPAYMENTS")
print(payments.columns.tolist())

ORDERS
['order_id', 'customer_id', 'order_status', 'order_purchase_timestamp', 'order_approved_at', 'order_delivered_carrier_date', 'order_delivered_customer_date', 'order_estimated_delivery_date']

CUSTOMERS
['customer_id', 'customer_unique_id', 'customer_zip_code_prefix', 'customer_city', 'customer_state']

ITEMS
['order_id', 'order_item_id', 'product_id', 'seller_id', 'shipping_limit_date', 'price', 'freight_value']

PAYMENTS
['order_id', 'payment_sequential', 'payment_type', 'payment_installments', 'payment_value']


In [7]:
customer_data = customers[
    [
        "customer_id",
        "customer_unique_id",
        "customer_city",
        "customer_state"
    ]
].copy()

print(customer_data.head())
print("Customer rows:", len(customer_data))

                        customer_id                customer_unique_id  \
0  06b8999e2fba1a1fbc88172c00ba8bc7  861eff4711a542e4b93843c6dd7febb0   
1  18955e83d337fd6b2def6b18a428ac77  290c77bc529b7ac935b93aa66c333dc3   
2  4e7b3e00288586ebd08712fdd0374a03  060e732b5b29e8181a18229c7b0b2b5e   
3  b2b6027bc5c5109e529d4dc6358b12c3  259dac757896d24d7702b9acbbff3f3c   
4  4f2d8ab171c80ec8364f7c12e35b23ad  345ecd01c38d18a9036ed96c73b8d066   

           customer_city customer_state  
0                 franca             SP  
1  sao bernardo do campo             SP  
2              sao paulo             SP  
3        mogi das cruzes             SP  
4               campinas             SP  
Customer rows: 99441


In [8]:
item_summary = (
    items
    .groupby("order_id")
    .agg(
        number_of_items=("order_item_id", "count"),
        number_of_products=("product_id", "nunique"),
        number_of_sellers=("seller_id", "nunique"),
        total_item_price=("price", "sum"),
        total_freight_value=("freight_value", "sum")
    )
    .reset_index()
)

print(item_summary.head())
print("Aggregated item records:", len(item_summary))

                           order_id  number_of_items  number_of_products  \
0  00010242fe8c5a6d1ba2dd792cb16214                1                   1   
1  00018f77f2f0320c557190d7a144bdd3                1                   1   
2  000229ec398224ef6ca0657da4fc703e                1                   1   
3  00024acbcdf0a6daa1e931b038114c75                1                   1   
4  00042b26cf59d7ce69dfabb4e55b4fd9                1                   1   

   number_of_sellers  total_item_price  total_freight_value  
0                  1             58.90                13.29  
1                  1            239.90                19.93  
2                  1            199.00                17.87  
3                  1             12.99                12.79  
4                  1            199.90                18.14  
Aggregated item records: 98666


In [9]:
payment_summary = (
    payments
    .groupby("order_id")
    .agg(
        payment_value=("payment_value", "sum"),
        payment_installments=("payment_installments", "max")
    )
    .reset_index()
)

print(payment_summary.head())
print("Aggregated payment records:", len(payment_summary))

                           order_id  payment_value  payment_installments
0  00010242fe8c5a6d1ba2dd792cb16214          72.19                     2
1  00018f77f2f0320c557190d7a144bdd3         259.83                     3
2  000229ec398224ef6ca0657da4fc703e         216.87                     5
3  00024acbcdf0a6daa1e931b038114c75          25.78                     2
4  00042b26cf59d7ce69dfabb4e55b4fd9         218.04                     3
Aggregated payment records: 99440


In [10]:
payment_type_summary = (
    payments
    .groupby("order_id")["payment_type"]
    .apply(lambda x: "|".join(sorted(x.dropna().astype(str).unique())))
    .reset_index(name="payment_type")
)

print(payment_type_summary.head())

                           order_id payment_type
0  00010242fe8c5a6d1ba2dd792cb16214  credit_card
1  00018f77f2f0320c557190d7a144bdd3  credit_card
2  000229ec398224ef6ca0657da4fc703e  credit_card
3  00024acbcdf0a6daa1e931b038114c75  credit_card
4  00042b26cf59d7ce69dfabb4e55b4fd9  credit_card


In [11]:
order_data = orders[
    [
        "order_id",
        "customer_id",
        "order_status",
        "order_purchase_timestamp",
        "order_approved_at",
        "order_delivered_carrier_date",
        "order_delivered_customer_date",
        "order_estimated_delivery_date"
    ]
].copy()

print(order_data.head())

                           order_id                       customer_id  \
0  e481f51cbdc54678b7cc49136f2d6af7  9ef432eb6251297304e76186b10a928d   
1  53cdb2fc8bc7dce0b6741e2150273451  b0830fb4747a6c6d20dea0b8c802d7ef   
2  47770eb9100c2d0c44946d9cf07ec65d  41ce2a54c0b03bf3443c3d931a367089   
3  949d5b44dbf5de918fe9c16f97b45f8a  f88197465ea7920adcdbec7375364d82   
4  ad21c59c0840e6cb83a9ceb5573f8159  8ab97904e6daea8866dbdbc4fb7aad2c   

  order_status order_purchase_timestamp    order_approved_at  \
0    delivered      2017-10-02 10:56:33  2017-10-02 11:07:15   
1    delivered      2018-07-24 20:41:37  2018-07-26 03:24:27   
2    delivered      2018-08-08 08:38:49  2018-08-08 08:55:23   
3    delivered      2017-11-18 19:28:06  2017-11-18 19:45:59   
4    delivered      2018-02-13 21:18:39  2018-02-13 22:20:29   

  order_delivered_carrier_date order_delivered_customer_date  \
0          2017-10-04 19:55:00           2017-10-10 21:25:13   
1          2018-07-26 14:31:00           2018-08

In [12]:
spending = order_data.merge(
    customer_data,
    on="customer_id",
    how="left"
)

spending = spending.merge(
    item_summary,
    on="order_id",
    how="left"
)

spending = spending.merge(
    payment_summary,
    on="order_id",
    how="left"
)

spending = spending.merge(
    payment_type_summary,
    on="order_id",
    how="left"
)

print("Final shape:", spending.shape)

Final shape: (99441, 19)


In [13]:
print(spending.head())

                           order_id                       customer_id  \
0  e481f51cbdc54678b7cc49136f2d6af7  9ef432eb6251297304e76186b10a928d   
1  53cdb2fc8bc7dce0b6741e2150273451  b0830fb4747a6c6d20dea0b8c802d7ef   
2  47770eb9100c2d0c44946d9cf07ec65d  41ce2a54c0b03bf3443c3d931a367089   
3  949d5b44dbf5de918fe9c16f97b45f8a  f88197465ea7920adcdbec7375364d82   
4  ad21c59c0840e6cb83a9ceb5573f8159  8ab97904e6daea8866dbdbc4fb7aad2c   

  order_status order_purchase_timestamp    order_approved_at  \
0    delivered      2017-10-02 10:56:33  2017-10-02 11:07:15   
1    delivered      2018-07-24 20:41:37  2018-07-26 03:24:27   
2    delivered      2018-08-08 08:38:49  2018-08-08 08:55:23   
3    delivered      2017-11-18 19:28:06  2017-11-18 19:45:59   
4    delivered      2018-02-13 21:18:39  2018-02-13 22:20:29   

  order_delivered_carrier_date order_delivered_customer_date  \
0          2017-10-04 19:55:00           2017-10-10 21:25:13   
1          2018-07-26 14:31:00           2018-08

In [14]:
print(spending.columns.tolist())

['order_id', 'customer_id', 'order_status', 'order_purchase_timestamp', 'order_approved_at', 'order_delivered_carrier_date', 'order_delivered_customer_date', 'order_estimated_delivery_date', 'customer_unique_id', 'customer_city', 'customer_state', 'number_of_items', 'number_of_products', 'number_of_sellers', 'total_item_price', 'total_freight_value', 'payment_value', 'payment_installments', 'payment_type']


In [15]:
print("Total rows:", len(spending))
print("Unique orders:", spending["order_id"].nunique())

Total rows: 99441
Unique orders: 99441


In [16]:
duplicate_orders = spending["order_id"].duplicated().sum()

print("Duplicate order rows:", duplicate_orders)

Duplicate order rows: 0


In [17]:
missing = spending.isnull().sum()

print(missing[missing > 0].sort_values(ascending=False))

order_delivered_customer_date    2965
order_delivered_carrier_date     1783
number_of_items                   775
total_freight_value               775
number_of_products                775
number_of_sellers                 775
total_item_price                  775
order_approved_at                 160
payment_value                       1
payment_installments                1
payment_type                        1
dtype: int64


In [18]:
date_columns = [
    "order_purchase_timestamp",
    "order_approved_at",
    "order_delivered_carrier_date",
    "order_delivered_customer_date",
    "order_estimated_delivery_date"
]

for col in date_columns:
    spending[col] = pd.to_datetime(spending[col], errors="coerce")

print(spending[date_columns].dtypes)

order_purchase_timestamp         datetime64[ns]
order_approved_at                datetime64[ns]
order_delivered_carrier_date     datetime64[ns]
order_delivered_customer_date    datetime64[ns]
order_estimated_delivery_date    datetime64[ns]
dtype: object


In [19]:
spending["purchase_year"] = spending["order_purchase_timestamp"].dt.year
spending["purchase_month"] = spending["order_purchase_timestamp"].dt.month
spending["purchase_dayofweek"] = spending["order_purchase_timestamp"].dt.dayofweek

print(
    spending[
        [
            "order_purchase_timestamp",
            "purchase_year",
            "purchase_month",
            "purchase_dayofweek"
        ]
    ].head()
)

  order_purchase_timestamp  purchase_year  purchase_month  purchase_dayofweek
0      2017-10-02 10:56:33           2017              10                   0
1      2018-07-24 20:41:37           2018               7                   1
2      2018-08-08 08:38:49           2018               8                   2
3      2017-11-18 19:28:06           2017              11                   5
4      2018-02-13 21:18:39           2018               2                   1


In [20]:
spending["purchase_year"] = spending["order_purchase_timestamp"].dt.year
spending["purchase_month"] = spending["order_purchase_timestamp"].dt.month
spending["purchase_dayofweek"] = spending["order_purchase_timestamp"].dt.dayofweek

print(
    spending[
        [
            "order_purchase_timestamp",
            "purchase_year",
            "purchase_month",
            "purchase_dayofweek"
        ]
    ].head()
)

  order_purchase_timestamp  purchase_year  purchase_month  purchase_dayofweek
0      2017-10-02 10:56:33           2017              10                   0
1      2018-07-24 20:41:37           2018               7                   1
2      2018-08-08 08:38:49           2018               8                   2
3      2017-11-18 19:28:06           2017              11                   5
4      2018-02-13 21:18:39           2018               2                   1


In [21]:
output_file = DATA_DIR / "olist_spending_dataset.csv"

spending.to_csv(output_file, index=False)

print("Saved successfully!")
print("File:", output_file)
print("Rows:", len(spending))
print("Columns:", len(spending.columns))

Saved successfully!
File: archive\olist_spending_dataset.csv
Rows: 99441
Columns: 22


In [22]:
final_data = pd.read_csv(output_file)

print("===================================")
print("FINAL SPENDING DATASET")
print("===================================")

print("Rows:", final_data.shape[0])
print("Columns:", final_data.shape[1])
print("Unique orders:", final_data["order_id"].nunique())
print("Duplicate orders:", final_data["order_id"].duplicated().sum())

print("\nColumns:")
for col in final_data.columns:
    print("-", col)

print("\nFirst 5 records:")
display(final_data.head())

FINAL SPENDING DATASET
Rows: 99441
Columns: 22
Unique orders: 99441
Duplicate orders: 0

Columns:
- order_id
- customer_id
- order_status
- order_purchase_timestamp
- order_approved_at
- order_delivered_carrier_date
- order_delivered_customer_date
- order_estimated_delivery_date
- customer_unique_id
- customer_city
- customer_state
- number_of_items
- number_of_products
- number_of_sellers
- total_item_price
- total_freight_value
- payment_value
- payment_installments
- payment_type
- purchase_year
- purchase_month
- purchase_dayofweek

First 5 records:


,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date,customer_unique_id,customer_city,...,number_of_products,number_of_sellers,total_item_price,total_freight_value,payment_value,payment_installments,payment_type,purchase_year,purchase_month,purchase_dayofweek
0,e481f51cbdc54678b7cc49136f2d6af7,9ef432eb6251297304e76186b10a928d,delivered,2017-10-02 10:56:33,2017-10-02 11:07:15,2017-10-04 19:55:00,2017-10-10 21:25:13,2017-10-18,7c396fd4830fd04220f754e42b4e5bff,sao paulo,...,1.0,1.0,29.99,8.72,38.71,1.0,credit_card|voucher,2017,10,0
1,53cdb2fc8bc7dce0b6741e2150273451,b0830fb4747a6c6d20dea0b8c802d7ef,delivered,2018-07-24 20:41:37,2018-07-26 03:24:27,2018-07-26 14:31:00,2018-08-07 15:27:45,2018-08-13,af07308b275d755c9edb36a90c618231,barreiras,...,1.0,1.0,118.70,22.76,141.46,1.0,boleto,2018,7,1
2,47770eb9100c2d0c44946d9cf07ec65d,41ce2a54c0b03bf3443c3d931a367089,delivered,2018-08-08 08:38:49,2018-08-08 08:55:23,2018-08-08 13:50:00,2018-08-17 18:06:29,2018-09-04,3a653a41f6f9fc3d2a113cf8398680e8,vianopolis,...,1.0,1.0,159.90,19.22,179.12,3.0,credit_card,2018,8,2
3,949d5b44dbf5de918fe9c16f97b45f8a,f88197465ea7920adcdbec7375364d82,delivered,2017-11-18 19:28:06,2017-11-18 19:45:59,2017-11-22 13:39:59,2017-12-02 00:28:42,2017-12-15,7c142cf63193a1473d2e66489a9ae977,sao goncalo do amarante,...,1.0,1.0,45.00,27.20,72.20,1.0,credit_card,2017,11,5
4,ad21c59c0840e6cb83a9ceb5573f8159,8ab97904e6daea8866dbdbc4fb7aad2c,delivered,2018-02-13 21:18:39,2018-02-13 22:20:29,2018-02-14 19:46:34,2018-02-16 18:17:02,2018-02-26,72632f0f9dd73dfee390c9b22eb56dd6,santo andre,...,1.0,1.0,19.90,8.72,28.62,1.0,credit_card,2018,2,1
